In [ ]:
# General
import numpy as np
import pandas as pd
import os
from tqdm import tqdm
import re
from datetime import date
import math
import seaborn as sns

# Plotting
import matplotlib.pyplot as plt
import plotly
import plotly.graph_objects as go
import plotly.express as px
import plotly.io as pio
from plotly.offline import plot
import plotly.subplots as sp
from plotly.subplots import make_subplots
pd.options.display.float_format = '{:.2f}'.format

In [ ]:
# Define user
user = os.getlogin()
path_users = os.path.join('C:\\Users', user)

# Set file paths
if user == 'jfontes':
    # Git
    path_git = os.path.join(path_users, 'Documents', 'Projects', 'Regional-Monitoring', 'Indicator_Gen')

path_config0 = os.path.join(path_git, 'config')


In [ ]:
## User defined functions
exec(open(os.path.join(path_config0, 'Functions.py')).read())

In [ ]:
# Set parameters for export file
path_plots = r"\\webmapping-svr\c$\inetpub\wwwroot\monitoring"
print('Export Location: ' + path_plots)

***

Monthly rent costs with Peer MSA

***

In [ ]:
## Importing ---

# Set Indicator
indicator_name = 'Cost_2'
plot_name = 'rent_cost_monthly'
export = False


file_name = f"{indicator_name} MPO Zillow.xlsx"
df_cost = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'Monthly')


## Organizing ---


df_plot = df_cost.copy()

list_regions = [
    'Sacramento, CA'
    , 'Yuba City, CA'
    , 'Austin, TX'
    , 'Charlotte, NC'
    , 'Cincinnati, OH'
    , 'Cleveland, OH'
    , 'Columbus, OH'
    , 'Detroit, MI'
    , 'Indianapolis, IN'
    , 'Los Angeles, CA'
    , 'Miami, FL'
    , 'Orlando, FL'
    , 'Portland, OR'
    , 'Riverside, CA'
    , 'Salt Lake City, UT'
    , 'San Antonio, TX'
    , 'San Diego, CA'
    , 'San Francisco, CA'
    , 'San Jose, CA'
    , 'St. Louis, MO'
    , 'Tampa, FL'
]
df_plot = df_plot[df_plot['Region'].isin(list_regions)]


df_plot['Sort'] = pd.Categorical(df_plot['Region'], list_regions)

df_plot = df_plot.sort_values(['Sort', 'date_'], ascending = [True, False])
df_plot = df_plot.drop(['Sort'], axis = 1)
df_plot = df_plot.dropna()

df_plot = df_plot.reset_index(drop=True)
df_plot['Price'] = round(df_plot['Price'])
df_plot = df_plot.dropna()

display(df_plot.head())


## Plotting ---


fig = px.line(df_plot, x='date_', y='Price', color='Region')

title = '<b>Monthly Sales Cost</b>'
fig.update_yaxes(tick0=0, dtick=500, tickprefix='$', tickformat = ',.0f')
fig.update_xaxes(dtick="M48", tickformat="%b\n%Y", ticklabelmode="period")
fig.update_traces(hovertemplate='%{y}')

plot_agol(export=export)

***

Monthly sales cost with Peer MSA

***

In [ ]:
## Importing ---

# Set Indicator
indicator_name = 'Cost_1'
plot_name = 'sales_cost_monthly'
export = False


file_name = f"{indicator_name} MPO Zillow.xlsx"
df_cost = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'Monthly')


## Organizing ---


df_plot = df_cost.copy()

list_regions = [
    'Sacramento'
    , 'Yuba City'
    , 'Austin'
    , 'Charlotte'
    , 'Cincinnati'
    , 'Cleveland'
    , 'Columbus'
    , 'Detroit'
    , 'Indianapolis'
    , 'Los Angeles'
    , 'Miami'
    , 'Orlando'
    , 'Portland'
    , 'Riverside'
    , 'Salt Lake City'
    , 'San Antonio'
    , 'San Diego'
    , 'San Francisco'
    , 'San Jose'
    , 'St. Louis'
    , 'Tampa'
]
df_plot = df_plot[df_plot['Region'].isin(list_regions)]


df_plot['Sort'] = pd.Categorical(df_plot['Region'], list_regions)

df_plot = df_plot.sort_values(['Sort', 'date_'], ascending = [True, False])
df_plot = df_plot.drop(['Sort'], axis = 1)
df_plot = df_plot.dropna()

df_plot = df_plot.reset_index(drop=True)
df_plot['Price'] = round(df_plot['Price'])
df_plot['Region'] = df_plot['Region'].map(peer_msa_labels)
df_plot = df_plot.dropna()

display(df_plot.head())


## Plotting ---


fig = px.line(df_plot, x='date_', y='Price', color='Region')

title = '<b>Monthly Sales Cost</b>'
fig.update_yaxes(tick0=0, dtick=200000, tickprefix='$', tickformat = ',.0f')
fig.update_xaxes(dtick="M48", tickformat="%b\n%Y", ticklabelmode="period")
fig.update_traces(hovertemplate='%{y}')

plot_agol(export=export)

***

Quarterly sales cost with Peer MSA

***

In [ ]:
## Importing ---

# Set Indicator
indicator_name = 'Cost_1'
plot_name = 'sales_cost_quarterly'
export = False


file_name = f"{indicator_name} MPO Zillow.xlsx"
df_cost = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'Monthly')


## Organizing ---


df_plot = df_cost.copy()

df_plot['date_'] = pd.to_datetime(df_plot['date_'])
df_plot['Year' ] = df_plot['date_'].dt.year
df_plot['Month'] = df_plot['date_'].dt.month


conditions = [
      df_plot['Month'].isin([1, 2, 3])
    , df_plot['Month'].isin([4, 5, 6])
    , df_plot['Month'].isin([7, 8, 9])
    , df_plot['Month'].isin([10, 11, 12])
]

choices = ['Q1', 'Q2', 'Q3', 'Q4']

df_plot['Quarter'] = np.select(conditions, choices)

df_plot['Year_Q'] = df_plot['Year'].astype(str) + '-' + df_plot['Quarter'].astype(str)
df_plot = df_plot.drop('date_', axis = 1)
df_plot = df_plot.groupby(['State', 'MPO', 'Region', 'Year_Q', 'Quarter'], as_index = False)['Price'].mean()
df_plot = df_plot.sort_values(by=['State', 'MPO', 'Region', 'Year_Q'], ascending = [True, True, True, True])


list_regions = [
    'Sacramento'
    , 'Yuba City'
    , 'Austin'
    , 'Charlotte'
    , 'Cincinnati'
    , 'Cleveland'
    , 'Columbus'
    , 'Detroit'
    , 'Indianapolis'
    , 'Los Angeles'
    , 'Miami'
    , 'Orlando'
    , 'Portland'
    , 'Riverside'
    , 'Salt Lake City'
    , 'San Antonio'
    , 'San Diego'
    , 'San Francisco'
    , 'San Jose'
    , 'St. Louis'
    , 'Tampa'
]
df_plot = df_plot[df_plot['Region'].isin(list_regions)]


df_plot['Sort'] = pd.Categorical(df_plot['Region'], list_regions)

df_plot = df_plot.sort_values(['Sort', 'Year_Q'], ascending = [True, True])
df_plot = df_plot.drop(['Sort'], axis = 1)
df_plot = df_plot.dropna()

df_plot = df_plot.reset_index(drop=True)


df_plot['Price'] = round(df_plot['Price'])
df_plot['Region'] = df_plot['Region'].map(peer_msa_labels)
df_plot = df_plot.dropna()

display(df_plot.head())


## Plotting ---


fig = px.line(df_plot, x='Year_Q', y='Price', color='Region')

title = '<b>Median Home Sale Price by Quarter</b>'
fig.update_yaxes(tick0=0, dtick=200000, tickprefix='$', tickformat = ',.0f')
# fig.update_xaxes(dtick="M48", tickformat="%b\n%Y", ticklabelmode="period")
fig.update_traces(hovertemplate='%{y}')

plot_agol(export=export)

***

Monthly growth rate in sales cost

***

In [ ]:
## Importing ---

# Set Indicator
indicator_name = 'Cost_1'
plot_name = 'sales_cost_monthly_gr'
export = False


file_name = f"{indicator_name} MPO Zillow.xlsx"
df_cost = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'Monthly')


## Organizing ---


df_plot = df_cost.copy()

df_plot['date_'] = pd.to_datetime(df_plot['date_'])
df_plot['Year' ] = df_plot['date_'].dt.year
df_plot['Month'] = df_plot['date_'].dt.month

df_plot = df_plot.sort_values(['State', 'MPO', 'Region', 'date_'], ascending = [True, True, True, True])
df_plot['Price_GR'] = df_plot['Price'].pct_change()*100
df_plot.loc[df_plot['date_'] == '2000-01-31', 'Price_GR'] = np.nan
df_plot.loc[df_plot['Price_GR'] == np.inf, 'Price_GR'] = np.nan
df_plot = df_plot.sort_values(['Region', 'date_'], ascending = [True, False])
df_plot['first_date'] = df_plot.groupby(['State', 'MPO', 'Region'], as_index = False)['date_'].transform('min')

df_plot = df_plot[df_plot['date_'] != df_plot['first_date']]
df_plot = df_plot[~df_plot['Price_GR'].isna()]
df_plot = df_plot[~df_plot['Price'].isna()]

df_plot = df_plot.groupby(['State', 'MPO', 'Region', 'Month'], as_index = False)['Price_GR'].mean()
df_plot = df_plot.sort_values(by=['State', 'MPO', 'Region', 'Month'], ascending = [True, True, True, True])

df_plot = df_plot.reset_index(drop = True)

list_regions = [
    'Sacramento'
    , 'Yuba City'
    , 'Austin'
    , 'Charlotte'
    , 'Cincinnati'
    , 'Cleveland'
    , 'Columbus'
    , 'Detroit'
    , 'Indianapolis'
    , 'Los Angeles'
    , 'Miami'
    , 'Orlando'
    , 'Portland'
    , 'Riverside'
    , 'Salt Lake City'
    , 'San Antonio'
    , 'San Diego'
    , 'San Francisco'
    , 'San Jose'
    , 'St. Louis'
    , 'Tampa'
]
df_plot = df_plot[df_plot['Region'].isin(list_regions)]


df_plot['Sort'] = pd.Categorical(df_plot['Region'], list_regions)

df_plot = df_plot.sort_values(['Sort', 'Month'], ascending = [True, True])
df_plot = df_plot.drop(['Sort'], axis = 1)
df_plot = df_plot.dropna()

df_plot = df_plot.reset_index(drop=True)

df_plot['Region'] = df_plot['Region'].map(peer_msa_labels)
df_plot = df_plot.dropna()

display(df_plot.head())


## Plotting ---


fig = px.line(df_plot, x='Month', y='Price_GR', color='Region')

title = '<b>Growth rate by month</b>'
# fig.update_yaxes(tick0=0, dtick=200000, tickprefix='$', tickformat = ',.0f')
# fig.update_xaxes(dtick="M48", tickformat="%b\n%Y", ticklabelmode="period")
fig.update_traces(hovertemplate='%{y}')

plot_agol(export=export)

***

Growth rate by quarter

***

In [ ]:
## Importing ---

# Set Indicator
indicator_name = 'Cost_1'
plot_name = 'sales_cost_quarterly'
export = False


file_name = f"{indicator_name} MPO Zillow.xlsx"
df_cost = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'Monthly')


## Organizing ---


df_plot = df_cost.copy()

df_plot['date_'] = pd.to_datetime(df_plot['date_'])
df_plot['Year' ] = df_plot['date_'].dt.year
df_plot['Month'] = df_plot['date_'].dt.month


df_plot = df_plot.sort_values(['State', 'MPO', 'Region', 'date_'], ascending = [True, True, True, True])
df_plot['Price_GR'] = df_plot['Price'].pct_change()*100
df_plot.loc[df_plot['date_'] == '2000-01-31', 'Price_GR'] = np.nan
df_plot.loc[df_plot['Price_GR'] == np.inf, 'Price_GR'] = np.nan
df_plot = df_plot.sort_values(['Region', 'date_'], ascending = [True, False])
df_plot['first_date'] = df_plot.groupby(['State', 'MPO', 'Region'], as_index = False)['date_'].transform('min')

df_plot = df_plot[df_plot['date_'] != df_plot['first_date']]
df_plot = df_plot[~df_plot['Price_GR'].isna()]
df_plot = df_plot[~df_plot['Price'].isna()]
df_plot = df_plot.reset_index(drop = True)


conditions = [
      df_plot['Month'].isin([1, 2, 3])
    , df_plot['Month'].isin([4, 5, 6])
    , df_plot['Month'].isin([7, 8, 9])
    , df_plot['Month'].isin([10, 11, 12])
]

choices = ['Q1', 'Q2', 'Q3', 'Q4']

df_plot['Quarter'] = np.select(conditions, choices)

df_plot = df_plot.groupby(['State', 'MPO', 'Region', 'Quarter'], as_index = False)['Price_GR'].mean()
df_plot = df_plot.sort_values(by=['State', 'MPO', 'Region', 'Quarter'], ascending = [True, True, True, True])


list_regions = [
    'Sacramento'
    , 'Yuba City'
    , 'Austin'
    , 'Charlotte'
    , 'Cincinnati'
    , 'Cleveland'
    , 'Columbus'
    , 'Detroit'
    , 'Indianapolis'
    , 'Los Angeles'
    , 'Miami'
    , 'Orlando'
    , 'Portland'
    , 'Riverside'
    , 'Salt Lake City'
    , 'San Antonio'
    , 'San Diego'
    , 'San Francisco'
    , 'San Jose'
    , 'St. Louis'
    , 'Tampa'
]
df_plot = df_plot[df_plot['Region'].isin(list_regions)]


df_plot['Sort'] = pd.Categorical(df_plot['Region'], list_regions)

df_plot = df_plot.sort_values(['Sort', 'Quarter'], ascending = [True, True])
df_plot = df_plot.drop(['Sort'], axis = 1)
df_plot = df_plot.dropna()

df_plot = df_plot.reset_index(drop=True)


df_plot['Region'] = df_plot['Region'].map(peer_msa_labels)
df_plot = df_plot.dropna()

display(df_plot.head())


## Plotting ---


fig = px.line(df_plot, x='Quarter', y='Price_GR', color='Region')

title = '<b>Median Home Sale Price by Quarter</b>'
fig.update_yaxes(tick0=0, dtick=200000, tickprefix='$', tickformat = ',.0f')
# fig.update_xaxes(dtick="M48", tickformat="%b\n%Y", ticklabelmode="period")
fig.update_traces(hovertemplate='%{y}')

plot_agol(export=export)

***

Housing cost to median household income ratio

***

In [ ]:

# Set Indicator
indicator_name = 'Income_1'

file_name = f"{indicator_name} MPO ACS5.xlsx"
df_inc = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MPO')
df_inc



In [ ]:
## Importing ---

# Set Indicator
indicator_name = 'Income_1'

file_name = f"{indicator_name} MPO ACS5.xlsx"
df_inc = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MPO')



# Set Indicator
indicator_name = 'Cost_1'
plot_name = 'cost_to_income_ratio'
export = False


file_name = f"{indicator_name} MPO Zillow.xlsx"
df_cost = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'Annual')


## Organizing ---

df_cost = df_cost[df_cost['Region'] == 'SACOG Median']
df_cost = df_cost[['Year', 'Price']]

df_cpi = pd.read_excel(os.path.join(path_git, 'config', 'CPI_IAF.xlsx'), sheet_name = 'BLS_West')
df_cpi = df_cpi[['Year', 'IAF_' + str(2023)]]

df_cost = df_cost.merge(df_cpi, on = 'Year', how = 'left')
df_cost['Price'] = round(df_cost['Price']*df_cost['IAF_' + str(2023)])
df_cost = df_cost.drop(['IAF_' + str(2023)], axis = 1)


df_inc = df_inc[['MPO', 'Year', 'Race_Ethnicity', 'Median Household Income']]


df_inc = df_inc.merge(df_cost, on='Year', how='left')
df_inc['Cost to Median Income Ratio'] = round(df_inc['Price']/df_inc['Median Household Income'], 2)


df_plot = df_inc[df_inc['Race_Ethnicity'].isin(['Black or African American', 'Asian', 'Hispanic or Latino', 'White (NH)'])]
df_inc = df_inc[df_inc['Race_Ethnicity'] == 'All']


df_plot['Sort'] = pd.Categorical(df_plot['Race_Ethnicity'], [
    'Asian'
    , 'Black or African American'
    , 'Hispanic or Latino'
    , 'White (NH)'
])
    
df_plot = df_plot.sort_values(['Year', 'Sort'], ascending=[False, True])
df_plot = df_plot.drop(['Sort'], axis = 1)
df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())



## Plotting ---


color_map  = {
    'Asian': '#9DC209'
    , 'Black or African American': '#1E90FF'
    , 'Hispanic or Latino': "#FBB117"
    , 'White (NH)': "#DC381F"
}


fig = px.line(df_plot, x='Year', y='Cost to Median Income Ratio', color='Race_Ethnicity', color_discrete_map=color_map, markers=True)

fig.add_trace(go.Scatter(x=df_inc["Year"], y=df_inc['Cost to Median Income Ratio']
                         , name = 'Average'
                         , line=go.scatter.Line(color="#2C3539", dash="dot")
                        ))

title = '<b>Housing Cost to Median Household Income Ratio</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=1, range = [0, 9.1])
fig.update_xaxes(tick0=0, dtick=1, range=[2008.5, 2023.5])
fig.update_traces(hovertemplate='%{y}')

plot_agol(export=export)



In [ ]:
## Importing ---

# Set Indicator
indicator_name = 'Income_1'

file_name = f"{indicator_name} MPO ACS5.xlsx"
df_inc = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'MPO')



# Set Indicator
indicator_name = 'Cost_2'
plot_name = 'rent_to_income_ratio'
export = False


file_name = f"{indicator_name} MPO Zillow.xlsx"
df_cost = pd.read_excel(os.path.join(path_plots, 'Data', file_name), sheet_name = 'Annual')


## Organizing ---

df_cost = df_cost[df_cost['Region'] == 'SACOG Median']
df_cost = df_cost[['Year', 'Price']]

df_cpi = pd.read_excel(os.path.join(path_git, 'config', 'CPI_IAF.xlsx'), sheet_name = 'BLS_West')
df_cpi = df_cpi[['Year', 'IAF_' + str(2023)]]

df_cost = df_cost.merge(df_cpi, on = 'Year', how = 'left')
df_cost['Price'] = round(df_cost['Price']*df_cost['IAF_' + str(2023)])
df_cost = df_cost.drop(['IAF_' + str(2023)], axis = 1)


df_inc = df_inc[['MPO', 'Year', 'Race_Ethnicity', 'Median Household Income']]


df_inc = df_inc.merge(df_cost, on='Year', how='left')
df_inc['Cost to Median Income Ratio'] = round(df_inc['Price']/(df_inc['Median Household Income']/12), 2)


df_plot = df_inc[df_inc['Race_Ethnicity'].isin(['Black or African American', 'Asian', 'Hispanic or Latino', 'White (NH)'])]
df_inc = df_inc[df_inc['Race_Ethnicity'] == 'All']


df_plot['Sort'] = pd.Categorical(df_plot['Race_Ethnicity'], [
    'Asian'
    , 'Black or African American'
    , 'Hispanic or Latino'
    , 'White (NH)'
])
    
df_plot = df_plot.sort_values(['Year', 'Sort'], ascending=[False, True])
df_plot = df_plot.drop(['Sort'], axis = 1)
df_plot = df_plot.reset_index(drop = True)

display(df_plot.head())



## Plotting ---


color_map  = {
    'Asian': '#9DC209'
    , 'Black or African American': '#1E90FF'
    , 'Hispanic or Latino': "#FBB117"
    , 'White (NH)': "#DC381F"
}


fig = px.line(df_plot, x='Year', y='Cost to Median Income Ratio', color='Race_Ethnicity', color_discrete_map=color_map, markers=True)

fig.add_trace(go.Scatter(x=df_inc["Year"], y=df_inc['Cost to Median Income Ratio']
                         , name = 'Average'
                         , line=go.scatter.Line(color="#2C3539", dash="dot")
                        ))

title = '<b>Rent to Monthly Median Household Income Ratio</b>  <br><sup>6-County Sacramento Region</sup>'
fig.update_yaxes(tick0=0, dtick=0.2, range = [0, 1.1])
fig.update_xaxes(tick0=0, dtick=1, range=[2014.5, 2022.5])
fig.update_traces(hovertemplate='%{y}')

plot_agol(export=export)